## Imports

In [1]:
library(reticulate)
library(iml)
library(counterfactuals)
library(data.table)

# IMPORTANT : pointer vers le python de TON environnement virtuel du projet
# (gradual-counterfactuals/venv), pas un python quelconque du systeme.
use_python("D:/M2/Soutenance/Experimentations/gradual-counterfactuals/venv/Scripts/python.exe", required = TRUE)
joblib <- import("joblib")

# On supprime le warning inoffensif "X does not have valid feature names"
# (le modele a ete entraine avec un DataFrame nomme, on lui passe ici une matrice)
py_run_string("import warnings; warnings.filterwarnings('ignore', message='.*valid feature names.*')")

cat("Bibliotheques importees.\n")

Bibliotheques importees.


## Paramètres

In [2]:
TARGET_COL     <- "price"
INDEX_INSTANCE <- 22L
DELTA          <- 1485.77

SEUIL_VALIDITE    <- 0.01
MARGE_GENERATION  <- 0.1
N_COUNTERFACTUALS <- 5L

## Chargement des fichiers

In [3]:
OUTPUT_DIR <- "../../outputs/automobile"

model_py <- joblib$load(file.path(OUTPUT_DIR, "svm_model.pkl"))

X_train <- fread(file.path(OUTPUT_DIR, "X_train.csv"), sep = ";")
y_train <- fread(file.path(OUTPUT_DIR, "y_train.csv"), sep = ";")[[TARGET_COL]]
X_test  <- fread(file.path(OUTPUT_DIR, "X_test.csv"), sep = ";")

exemple <- X_test[INDEX_INSTANCE, ]
cols    <- names(exemple)

cat(sprintf("[WhatIf] %d lignes de train chargees, instance %d du test set selectionnee.\n",
            nrow(X_train), INDEX_INSTANCE-1))

[WhatIf] 128 lignes de train chargees, instance 21 du test set selectionnee.


## Fonction de prédiction

In [4]:
predict_fun <- function(model, newdata) {
  X_np  <- r_to_py(as.matrix(newdata))
  preds <- model$predict(X_np)
  data.frame(price = as.numeric(py_to_r(preds)))
}

fx      <- predict_fun(model_py, exemple)$price[1]
Y_CIBLE <- fx + DELTA

cat(sprintf("[WhatIf] Prediction f(x) : %.2f\n", fx))
cat(sprintf("[WhatIf] Cible y'        : %.2f\n", Y_CIBLE))

[WhatIf] Prediction f(x) : 10514.23
[WhatIf] Cible y'        : 12000.00


## Construction du Predictor (iml) et lancement de What-If

In [5]:
if (Y_CIBLE > fx) {
  SENS <- "+"
} else if (Y_CIBLE < fx) {
  SENS <- "-"
} else {
  stop("Y_CIBLE est egal a f(x) : aucune modification n'est necessaire.")
}

marge         <- MARGE_GENERATION * abs(Y_CIBLE)
epsilon_marge <- 1e-2

if (SENS == "+") {
  borne_inf <- max(Y_CIBLE - marge, fx + epsilon_marge)
  borne_sup <- Y_CIBLE + marge
} else {
  borne_inf <- Y_CIBLE - marge
  borne_sup <- min(Y_CIBLE + marge, fx - epsilon_marge)
}

desired_outcome <- c(borne_inf, borne_sup)
cat(sprintf("[WhatIf] f(x) = %.2f | desired_outcome = [%.2f, %.2f]\n", fx, desired_outcome[1], desired_outcome[2]))

predictor <- Predictor$new(
  model             = model_py,
  data              = X_train,
  y                 = y_train,
  predict.function  = predict_fun
)

cat("[WhatIf] Recherche des plus proches voisins valides...\n")
wi_regr <- WhatIfRegr$new(predictor, n_counterfactuals = N_COUNTERFACTUALS)

cfactuals <- wi_regr$find_counterfactuals(
  x_interest      = as.data.frame(exemple),
  desired_outcome = desired_outcome
)

cat(sprintf("[WhatIf] %d contrefactuel(s) trouve(s).\n", nrow(cfactuals$data)))
cfactuals$data

[WhatIf] f(x) = 10514.23 | desired_outcome = [10800.00, 13200.00]
[WhatIf] Recherche des plus proches voisins valides...
[WhatIf] 5 contrefactuel(s) trouve(s).


normalized_losses,wheel_base,length,width,height,curb_weight,engine_size,bore,stroke,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>
104,99.1,186.6,66.5,56.1,2695,121,3.54,3.07,9.3,110,5250,21,28
164,99.8,176.6,66.2,54.3,2337,109,3.19,3.40,10.0,102,5500,24,30
104,99.1,186.6,66.5,56.1,2758,121,3.54,3.07,9.3,110,5250,21,28
134,98.4,176.2,65.6,52.0,2540,146,3.62,3.50,9.3,116,4800,24,30
134,98.4,176.2,65.6,52.0,2551,146,3.62,3.50,9.3,116,4800,24,30


## Affichage au format comparable

In [6]:
cf_df  <- as.data.frame(cfactuals$data)
x_orig <- as.data.frame(exemple)[1, ]

rows <- list()

ligne_orig <- c(motif = "-", nb_sauts = "-", support = "-", pred_cible = "-")
for (col in cols) ligne_orig[col] <- round(x_orig[[col]], 4)
ligne_orig["price"] <- round(fx, 2)
rows[[1]] <- ligne_orig

for (i in seq_len(nrow(cf_df))) {
  pred_cf <- predict_fun(model_py, cf_df[i, cols, drop = FALSE])$price[1]

  ligne <- c(motif = "WhatIf (kNN)", nb_sauts = "-", support = "-",
             pred_cible = paste0("[", round(desired_outcome[1],2), ", ", round(desired_outcome[2],2), "]"))
  for (col in cols) {
    val_cf <- cf_df[i, col]
    if (abs(x_orig[[col]] - val_cf) > 1e-6) {
      fleche <- if (val_cf > x_orig[[col]]) "↑" else "↓"
      ligne[col] <- paste0(round(val_cf, 4), " (", fleche, ")")
    } else {
      ligne[col] <- "-"
    }
  }
  fleche_price <- if (pred_cf > fx) "↑" else "↓"
  ligne["price"] <- paste0(round(pred_cf, 2), " (", fleche_price, ")")
  rows[[length(rows) + 1]] <- ligne
}

df_cf_whatif <- as.data.frame(do.call(rbind, rows), stringsAsFactors = FALSE)
fwrite(df_cf_whatif, file.path(OUTPUT_DIR, "contrefactuels_whatif.csv"), sep = ";")

cat("[WhatIf] Resultats sauvegardes dans contrefactuels_whatif.csv\n")
print(df_cf_whatif)

[WhatIf] Resultats sauvegardes dans contrefactuels_whatif.csv
         motif nb_sauts support     pred_cible normalized_losses wheel_base
1            -        -       -              -               110      103.3
2 WhatIf (kNN)        -       - [10800, 13200]           104 (↓)   99.1 (↓)
3 WhatIf (kNN)        -       - [10800, 13200]           164 (↑)   99.8 (↓)
4 WhatIf (kNN)        -       - [10800, 13200]           104 (↓)   99.1 (↓)
5 WhatIf (kNN)        -       - [10800, 13200]           134 (↑)   98.4 (↓)
6 WhatIf (kNN)        -       - [10800, 13200]           134 (↑)   98.4 (↓)
     length    width   height curb_weight engine_size     bore   stroke
1     174.6     64.6     59.8        2535         122     3.34     3.46
2 186.6 (↑) 66.5 (↑) 56.1 (↓)    2695 (↑)     121 (↓) 3.54 (↑) 3.07 (↓)
3 176.6 (↑) 66.2 (↑) 54.3 (↓)    2337 (↓)     109 (↓) 3.19 (↓)  3.4 (↓)
4 186.6 (↑) 66.5 (↑) 56.1 (↓)    2758 (↑)     121 (↓) 3.54 (↑) 3.07 (↓)
5 176.2 (↑) 65.6 (↑)   52 (↓)    2540 (↑)     

## Métriques

In [7]:
calculer_metriques <- function(x_orig, contrefactuels_list, fx_cible, seuil_validite = 0.01, mad) {
  n <- length(contrefactuels_list)
  if (n == 0) return(NULL)

  nb_valides <- sum(sapply(contrefactuels_list, function(cf) {
    abs(cf$pred_cf - fx_cible) < seuil_validite * abs(fx_cible)
  }))
  validite <- nb_valides / n

  proximites <- sapply(contrefactuels_list, function(cf) {
    mean(abs(unlist(x_orig) - unlist(cf$x_cf)) / mad)
  })
  proximite <- mean(proximites)

  nb_cols <- length(x_orig)
  sparsite_vals <- sapply(contrefactuels_list, function(cf) {
    nb_modifiees <- sum(abs(unlist(x_orig) - unlist(cf$x_cf)) > 1e-6)
    1 - nb_modifiees / nb_cols
  })
  sparsite <- mean(sparsite_vals)

  if (n < 2) {
    diversite <- 0
  } else {
    paires <- c()
    for (i in 1:(n - 1)) {
      for (j in (i + 1):n) {
        d <- mean(abs(unlist(contrefactuels_list[[i]]$x_cf) - unlist(contrefactuels_list[[j]]$x_cf)) / mad)
        paires <- c(paires, d)
      }
    }
    diversite <- mean(paires)
  }

  list(nb = n, validite = round(validite, 4), proximite = round(proximite, 4),
       sparsite = round(sparsite, 4), diversite = round(diversite, 4))
}

mad <- sapply(x_orig[cols], function(v) {
  m <- median(abs(v - median(v)))
  if (m == 0) 1 else m
})

contrefactuels_list <- lapply(seq_len(nrow(cf_df)), function(i) {
  pred_cf <- predict_fun(model_py, cf_df[i, cols, drop = FALSE])$price[1]
  list(x_cf = cf_df[i, cols], pred_cf = pred_cf)
})

m <- calculer_metriques(x_orig[cols], contrefactuels_list, Y_CIBLE, SEUIL_VALIDITE, mad)

cat(strrep("=", 50), "\n")
cat("  METRIQUES DU CONTREFACTUEL (WhatIf)\n")
cat(strrep("-", 50), "\n")
for (k in names(m)) cat(sprintf("  %-12s: %s\n", k, m[[k]]))

  METRIQUES DU CONTREFACTUEL (WhatIf)
-------------------------------------------------- 
  nb          : 5
  validite    : 0
  proximite   : 34.2361
  sparsite    : 0.0857
  diversite   : 45.6904
